<a href="https://colab.research.google.com/github/madelsu/MOSAIC-Agentic-Severity-Phenotyping/blob/main/Workstream_B/FRAMEWORK_MOSAIC_T2D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 📦 CELL 1: Unified Dependency Installation
# ============================================================

# We combine everything into one command so pip can find a compatible version of openai.
# 'crewai[tools]' already includes crewai, so we don't need to list both.
!pip install -U "crewai[tools]" litellm anthropic duckduckgo-search openpyxl tavily-python -q

# IMPORTANT: Force a specific OpenAI version that balances the two if conflicts persist
!pip install "openai>=1.83.0,<2.0.0" -q
!pip install  crewai tavily-python   -q

In [ ]:
# ============================================================
# 🔑 CELL 2 UPDATED: API Keys + Model Config
# ============================================================

import os
from openai import OpenAI
from anthropic import Anthropic
from crewai_tools import SerperDevTool
import os

os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["DEEPSEEK_API_KEY"] = ""
os.environ["SERPER_API_KEY"]    = ""

import os
from openai import OpenAI
from anthropic import Anthropic
from crewai_tools import SerperDevTool

# ── Model config ──────────────────────────────────────────────
# We now use GPT-4o for one of the assessors as requested.

ASSESSOR_1_MODEL   = "deepseek-chat"      # DeepSeek instance
ASSESSOR_2_MODEL   = "gpt-4o"            # OpenAI instance (Updated from GPT5.2)
CONSOLIDATOR_MODEL = "claude-opus-4-6" # Stable Claude version
EXTRACTOR_MODEL    = "claude-sonnet-4-6"    # Cheaper extraction

ASSESSOR_1_LABEL   = "DeepSeek-Assessor"
ASSESSOR_2_LABEL   = "GPT4o-Assessor"
CONSOLIDATOR_LABEL = "Claude-Consolidator"
EXTRACTOR_LABEL    = "Claude-Extractor"

print(f"✅ Configuration Loaded:")
print(f" 📌 Assessor 1: {ASSESSOR_1_MODEL}")
print(f" 📌 Assessor 2: {ASSESSOR_2_MODEL}")

# ── Test OpenAI (New) ──────────────────────────────────────────
print("\n--- 🧪 Testing OpenAI (Assessor 2) ---")
try:
    # Uses default base_url for OpenAI
    client_openai = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    r_oa = client_openai.chat.completions.create(
        model=ASSESSOR_2_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ OpenAI works! → {r_oa.choices[0].message.content}")
except Exception as e:
    print(f"❌ OpenAI failed: {e}")

# ── Test DeepSeek ─────────────────────────────────────────────
print("\n--- 🧪 Testing DeepSeek (Assessor 1) ---")
try:
    # Explicitly set DeepSeek base_url
    client_deepseek = OpenAI(
        api_key=os.environ["DEEPSEEK_API_KEY"],
        base_url="https://api.deepseek.com"
    )
    r_ds = client_deepseek.chat.completions.create(
        model=ASSESSOR_1_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ DeepSeek works! → {r_ds.choices[0].message.content}")
except Exception as e:
    print(f"❌ DeepSeek failed: {e}")

# ── Test Claude ───────────────────────────────────────────────
print("\n--- 🧪 Testing Claude (Consolidator) ---")
try:
    client_claude = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    r_c = client_claude.messages.create(
        model=CONSOLIDATOR_MODEL,
        max_tokens=10,
        messages=[{"role": "user", "content": "Hi"}]
    )
    print(f"✅ Claude works! → {r_c.content[0].text}")
except Exception as e:
    print(f"❌ Claude failed: {e}")

# ── Test Serper ───────────────────────────────────────────────
print("\n--- 🧪 Testing Serper Web Search ---")
try:
    search_tool = SerperDevTool()
    # CrewAI tools usually use .run() or ._run() depending on version
    result = search_tool.run(search_query="Type 2 diabetes severity clinical guidelines")
    print(f"✅ Serper works! Preview: {str(result)[:100]}...")
except Exception as e:
    print(f"❌ Serper failed: {e}")

print("\n" + "=" * 50)
print("🚀 All APIs tested and routing confirmed!")
print("=" * 50)

In [ ]:
# =============================================================================
#  CELL 3: UPLOADING 100 PATIENT FILE + SELECTION OF ONE PATIENT FOR PILOT MODE
# =============================================================================

import os
import json
import pandas as pd
from google.colab import files

# ── Upload file ───────────────────────────────────────────────
print("📤 Please upload your pipeline file...")
uploaded = files.upload()

# ── Load and verify ───────────────────────────────────────────
PIPELINE_FILE = list(uploaded.keys())[0]
xl = pd.ExcelFile(PIPELINE_FILE)

# Create a dictionary to map sheet names to your variables
# This makes it easy to loop through them for the summary
sheets_to_load = [
    "patient_list", "patients", "conditions", "observations",
    "medications", "encounters", "careplans", "allergies",
    "procedures", "devices", "immunizations"
]

data_frames = {}

print(f"\n--- Summary for {PIPELINE_FILE} ---")

for sheet in sheets_to_load:
    # Load the sheet
    df = pd.read_excel(PIPELINE_FILE, sheet_name=sheet)
    data_frames[sheet] = df # Store in dict for easy access

    # Print the requested details
    print(f"📄 Sheet: **{sheet}**")
    print(f"   - Columns ({len(df.columns)}): {', '.join(df.columns.tolist())}")

    if not df.empty:
        # We use .head(1) to get the first row as a pretty string
        print(f"   - Example Row:\n{df.iloc[[0]].to_string(index=False)}\n")
    else:
        print("   - Example Row: (Empty Sheet)\n")

# Load all sheets
patient_list  = pd.read_excel(PIPELINE_FILE, sheet_name="patient_list")
patients      = pd.read_excel(PIPELINE_FILE, sheet_name="patients")
conditions    = pd.read_excel(PIPELINE_FILE, sheet_name="conditions")
observations  = pd.read_excel(PIPELINE_FILE, sheet_name="observations")
medications   = pd.read_excel(PIPELINE_FILE, sheet_name="medications")
encounters    = pd.read_excel(PIPELINE_FILE, sheet_name="encounters")
careplans     = pd.read_excel(PIPELINE_FILE, sheet_name="careplans")
allergies     = pd.read_excel(PIPELINE_FILE, sheet_name="allergies")
procedures    = pd.read_excel(PIPELINE_FILE, sheet_name="procedures")
devices       = pd.read_excel(PIPELINE_FILE, sheet_name="devices")
immunizations = pd.read_excel(PIPELINE_FILE, sheet_name="immunizations")



# 🎯 LIMIT TO 1 PATIENT FOR PILOT
ALL_PATIENT_IDS = patient_list["PATIENT_ID"].tolist()
PILOT_PATIENT_ID = ALL_PATIENT_IDS[0] # Selects only the first ID
PATIENT_IDS = [PILOT_PATIENT_ID]      # Overwrites list to contain only 1

print(f"\n✅ Data loaded successfully!")
print(f" 🎯 PILOT MODE: Limited to 1 patient ID: {PILOT_PATIENT_ID}")

# ── Verify no ground truth columns leaked ────────────────────
forbidden = ["RULE_LABEL","SEVERITY_SCORE","SELECTION_REASON"]
leaked = [c for sheet in xl.sheet_names
          for c in pd.read_excel(PIPELINE_FILE, sheet_name=sheet).columns
          if any(f in str(c) for f in forbidden)]

if leaked:
    print(f" ❌ WARNING — leaked columns: {leaked}")
else:
    print(f" ✅ Clean — No ground truth leakage detected.")

# ── Output directory structure (OpenAI Included) ──────────────
OUTPUT_DIR = "/content/pipeline_outputs"
# Included phase1/openai and phase2/openai to match your dual-assessor setup
subdirs = [
    "phase1/openai", "phase1/deepseek", "phase1/critique",
    "phase1/consolidated_framework", "phase2/summaries",
    "phase2/openai", "phase2/deepseek", "phase2/consolidated",
    "phase3", "evaluation", "logs"
]

for subdir in subdirs:
    os.makedirs(f"{OUTPUT_DIR}/{subdir}", exist_ok=True)

print(f"📁 Output directories ready (including OpenAI paths)")

# ── Utility: save outputs ────────────────────────────────────
def save_output(phase_subdir, filename, content):
    """Saves to JSON and TXT for easy inspection."""
    base = f"{OUTPUT_DIR}/{phase_subdir}/{filename}"
    if isinstance(content, (dict, list)):
        with open(base + ".json", "w") as f:
            json.dump(content, f, indent=2)
    with open(base + ".txt", "w") as f:
        f.write(json.dumps(content, indent=2) if isinstance(content, (dict, list)) else str(content))

print("\n" + "=" * 50)
print(f"🚀 READY: Processing {len(PATIENT_IDS)} patient using OpenAI + DeepSeek.")
print("=" * 50)

In [ ]:
# ============================================================
# 🔧 CELL 4: PHASE 1 - Tavily + CrewAI setup and test
# T2D literature search with full page content
# ============================================================

import os
from tavily import TavilyClient
from crewai import Agent, Task, Crew
from crewai.tools import tool

# ── 1. API Keys Setup ─────────────────────────────────────────
# Make sure all three keys are loaded. In CrewAI/LiteLLM, Anthropic and
# DeepSeek models need their specific environment variables set.
os.environ["TAVILY_API_KEY"] = ""
os.environ["OPENAI_API_KEY"] = ""
os.environ["ANTHROPIC_API_KEY"] = ""
os.environ["DEEPSEEK_API_KEY"] = ""

# ── 2. Custom Tavily Tool ─────────────────────────────────────
@tool("Deep Medical Literature Search")
def medical_literature_search(query: str) -> str:
    """
    Searches the web for clinical guidelines, textbooks, and medical literature.
    Returns the full, raw text content of the top results for deep analysis.
    Use this when you need detailed clinical criteria, grading scales, or tables.
    """
    client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])
    response = client.search(
        query=query,
        max_results=3,
        search_depth="advanced",
        include_raw_content=True
    )

    results_str = ""
    for result in response.get("results", []):
        results_str += f"Title: {result['title']}\nURL: {result['url']}\n"
        content = result.get('raw_content', result['content'])
        results_str += f"Content:\n{content[:15000]}\n\n---\n\n"

    return results_str

# ── 3. Define the Agents (Using LiteLLM string format) ────────
print("\n🔵 Initializing Parallel Agents...")

# Assessor 1: GPT-4o
researcher_gpt = Agent(
    role='Senior Clinical Researcher (OpenAI)',
    goal='Find precise clinical criteria for classifying T2D severity.',
    backstory='You are a methodical medical researcher specializing in EHR phenotyping.',
    tools=[medical_literature_search],
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# Assessor 2: DeepSeek
# Note: "deepseek/" prefix tells litellm to route to the DeepSeek API
researcher_deepseek = Agent(
    role='Senior Clinical Researcher (DeepSeek)',
    goal='Find precise clinical criteria for classifying T2D severity.',
    backstory='You are a methodical medical researcher specializing in EHR phenotyping.',
    tools=[medical_literature_search],
    verbose=True,
    allow_delegation=False,
    llm="deepseek/deepseek-chat"
)

# Consolidator: Claude 3.5 Sonnet
# Note: "anthropic/" prefix tells litellm to route to the Anthropic API
consolidator_claude = Agent(
    role='Chief Medical Informatician',
    goal='Synthesize multiple research reports into a single, definitive, operational EHR phenotyping framework.',
    backstory='You are a senior physician and informatician. You excel at taking different proposed clinical criteria, resolving discrepancies, and formatting them into strict rules.',
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

# ── 4. Define the Tasks ───────────────────────────────────────
print("🔵 Defining Tasks...")

task_description = 'Search for T2D severity phenotyping criteria considering comorbidities and natural history. Return a bulleted list of the top most important criteria for which there is medical consensus. Make them operatable with specific thresholds.'
task_expected_output = 'A brief summary of clinical criteria for T2D severity along with references, optimized for EHR.'

# Task 1: GPT-4o Search (Runs asynchronously)
task_gpt = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_gpt,
    async_execution=True # <--- Key for parallel execution
)

# Task 2: DeepSeek Search (Runs asynchronously)
task_deepseek = Task(
    description=task_description,
    expected_output=task_expected_output,
    agent=researcher_deepseek,
    async_execution=True # <--- Key for parallel execution
)

# Task 3: Claude Consolidation (Runs sequentially after the first two)
task_consolidate = Task(
    description='Review the research provided by the two clinical researchers. Compare their findings, identify the strongest consensus, and produce a final, unified framework for T2D severity phenotyping.',
    expected_output='A definitive, consolidated guide of operatable EHR criteria for T2D severity with specific thresholds and references.',
    agent=consolidator_claude,
    context=[task_gpt, task_deepseek] # <--- Feeds the output of both previous tasks to Claude
)

# ── 5. Run the Crew ───────────────────────────────────────────
print("🚀 Kickoff CrewAI Parallel Execution...")

crew = Crew(
    agents=[researcher_gpt, researcher_deepseek, consolidator_claude],
    tasks=[task_gpt, task_deepseek, task_consolidate],
    verbose=True
)

import sys
import io

# ── Capture ALL console output ────────────────────────────────
log_capture = io.StringIO()

class TeeOutput:
    """Writes to both console AND a string buffer."""
    def __init__(self, *streams):
        self.streams = streams
    def write(self, text):
        for s in self.streams:
            s.write(text)
    def flush(self):
        for s in self.streams:
            s.flush()

# Start capturing
original_stdout = sys.stdout
sys.stdout = TeeOutput(original_stdout, log_capture)

try:
    result = crew.kickoff()

    # Stop capturing
    sys.stdout = original_stdout
    full_log = log_capture.getvalue()

    print("\n✅ CrewAI Execution Complete!")
    print(f"📊 Full log captured: {len(full_log)} chars")
    print("\n================ FINAL CONSOLIDATED OUTPUT ================\n")
    print(result)

    # ── Save everything ───────────────────────────────────────
    from google.colab import files
    import shutil

    # 1. Save the FULL verbose log (everything on screen)
    with open(f"{OUTPUT_DIR}/phase1/full_execution_log.txt", "w") as f:
        f.write(full_log)

    # 2. Save consolidated framework
    with open(f"{OUTPUT_DIR}/phase1/consolidated_framework/consolidated_framework.txt", "w") as f:
        f.write(str(result))

    # 3. Save individual agent final outputs
    for task_obj, name in [(task_gpt, "openai"), (task_deepseek, "deepseek"), (task_consolidate, "consolidated_framework")]:
        try:
            with open(f"{OUTPUT_DIR}/phase1/{name}/final_output.txt", "w") as f:
                f.write(str(task_obj.output))
            print(f"✅ Saved {name}: {len(str(task_obj.output))} chars")
        except Exception as e:
            print(f"⚠️ Could not save {name}: {e}")

    # 4. Zip and download
    shutil.make_archive("/content/phase1_outputs", "zip", f"{OUTPUT_DIR}/phase1")
    files.download("/content/phase1_outputs.zip")
    print("📥 Downloading!")

except Exception as e:
    sys.stdout = original_stdout
    print(f"❌ CrewAI failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================================
# 🔧 CELL 5: Phase 2 — Patient Severity Classification (PILOT: 1 Patient)(VERSION 1)
#
# ARCHITECTURE (4 agents):
#   Step 1: Claude Sonnet EXTRACTOR — framework-guided extraction of raw EHR
#   (NO COMPRESSION/KEYWORD EXTRACTION FOR PATIENT RAW INFO)
#   (WORKS FOR SOME PATIENTS BUT NOT FOR VERY LARGE PATIENTS)
#   Step 2: GPT-4o + DeepSeek ASSESSORS (parallel) — classify from summary
#   Step 3: Claude Sonnet CONSOLIDATOR — resolve disagreements → final tier
# ============================================================

from crewai import Agent, Task, Crew
import pandas as pd

# ── 1. Load Consolidated Framework from memory ────────────────
CONSOLIDATED_FRAMEWORK = str(result)
print(f"✅ Loaded consolidated framework from memory ({len(CONSOLIDATED_FRAMEWORK)} chars)")

# ── 2. Extract RAW Patient Data (minimal cleanup only) ───────
def extract_raw_patient_data(patient_id):
    """
    Extract raw patient data with only cosmetic cleanup:
    - Drop columns that are never clinically relevant (SSN, UDI, costs, payer)
    - Keep everything else for the Extractor LLM to intelligently process
    """
    sections = {}

    # Demographics — drop PII and financial columns
    demo = patients[patients["Id"] == patient_id]
    if not demo.empty:
        drop_cols = ["Id", "SSN", "DRIVERS", "PASSPORT", "ADDRESS", "CITY", "STATE",
                     "COUNTY", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE"]
        keep = [c for c in demo.columns if c not in drop_cols]
        sections["demographics"] = demo[keep].to_string(index=False)

    # Conditions — drop ENCOUNTER ID (not useful), keep everything else
    cond = conditions[conditions["PATIENT"] == patient_id]
    if not cond.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in cond.columns if c not in drop_cols]
        sections["conditions"] = cond[keep].to_string(index=False)

    # Observations — drop ENCOUNTER, PATIENT, TYPE
    obs = observations[observations["PATIENT"] == patient_id]
    if not obs.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "TYPE"]
        keep = [c for c in obs.columns if c not in drop_cols]
        sections["observations"] = f"[{len(obs)} total observation rows]\n" + obs[keep].to_string(index=False)

    # Medications — drop cost/payer columns
    meds = medications[medications["PATIENT"] == patient_id]
    if not meds.empty:
        drop_cols = ["PATIENT", "PAYER", "ENCOUNTER", "BASE_COST", "PAYER_COVERAGE", "DISPENSES", "TOTALCOST"]
        keep = [c for c in meds.columns if c not in drop_cols]
        sections["medications"] = f"[{len(meds)} total medication rows]\n" + meds[keep].to_string(index=False)

    # Encounters — drop cost/payer, keep class and reason
    enc = encounters[encounters["PATIENT"] == patient_id]
    if not enc.empty:
        drop_cols = ["Id", "PATIENT", "ORGANIZATION", "PROVIDER", "PAYER",
                     "BASE_ENCOUNTER_COST", "TOTAL_CLAIM_COST", "PAYER_COVERAGE"]
        keep = [c for c in enc.columns if c not in drop_cols]
        sections["encounters"] = f"[{len(enc)} total encounter rows]\n" + enc[keep].to_string(index=False)

    # Careplans
    cp = careplans[careplans["PATIENT"] == patient_id]
    if not cp.empty:
        drop_cols = ["Id", "PATIENT", "ENCOUNTER"]
        keep = [c for c in cp.columns if c not in drop_cols]
        sections["careplans"] = cp[keep].to_string(index=False)

    # Procedures
    proc = procedures[procedures["PATIENT"] == patient_id]
    if not proc.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "BASE_COST"]
        keep = [c for c in proc.columns if c not in drop_cols]
        sections["procedures"] = proc[keep].to_string(index=False)

    # Allergies
    allg = allergies[allergies["PATIENT"] == patient_id]
    if not allg.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in allg.columns if c not in drop_cols]
        sections["allergies"] = allg[keep].to_string(index=False)

    # Devices
    dev = devices[devices["PATIENT"] == patient_id]
    if not dev.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "UDI"]
        keep = [c for c in dev.columns if c not in drop_cols]
        sections["devices"] = dev[keep].to_string(index=False)

    # Immunizations — skip entirely (not relevant for T2D)
    sections["immunizations"] = "[Omitted — not relevant for T2D severity]"

    return sections

print(f"\n🔵 Extracting raw data for patient: {PILOT_PATIENT_ID}")
raw_data = extract_raw_patient_data(PILOT_PATIENT_ID)

raw_patient_record = f"=== RAW PATIENT RECORD: {PILOT_PATIENT_ID} ===\n\n"
for section, content in raw_data.items():
    raw_patient_record += f"--- {section.upper()} ---\n{content}\n\n"

print(f"✅ Raw patient data: {len(raw_patient_record)} chars (~{len(raw_patient_record)//4} tokens)")

# Save raw extraction for audit
save_output("phase2/summaries", f"patient_{PILOT_PATIENT_ID[:8]}_raw", raw_patient_record)

# ── 3. Define Agents ──────────────────────────────────────────
print("\n🔵 Initializing Phase 2 Agents (4-agent architecture)...")

# EXTRACTOR: Claude Sonnet — has 200K context, can handle the raw data
extractor = Agent(
    role='Clinical Data Extractor',
    goal='Extract and organize all clinically relevant information from raw EHR data, guided by the severity phenotyping framework. Produce a structured clinical summary that assessors can use for classification.',
    backstory=(
        'You are a clinical data specialist who excels at reading raw electronic health records '
        'and extracting the specific data points needed for clinical phenotyping. '
        'You are thorough — you never miss relevant data — but you also filter out noise. '
        'You organize your output by clinical domain so that downstream assessors can '
        'efficiently evaluate each criterion in the severity framework.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

# ASSESSOR 1: GPT-4o
assessor_gpt = Agent(
    role='Dr. A — Clinical Informatician',
    goal='Classify this T2D patient severity tier using the provided framework. Be methodical, data-driven, and conservative. Only classify based on evidence present in the extracted summary.',
    backstory=(
        'You are a clinical informatician specializing in EHR-based phenotyping. '
        'You are methodical and conservative — you only count what is explicitly documented. '
        'If data is missing or ambiguous, you note it but do NOT assume severity. '
        'You believe in strict adherence to framework criteria.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# ASSESSOR 2: DeepSeek
assessor_deepseek = Agent(
    role='Dr. B — Consultant Endocrinologist',
    goal='Classify this T2D patient severity tier using the provided framework. Take a holistic, patient-outcomes focused approach.',
    backstory=(
        'You are a consultant endocrinologist with 20 years of clinical experience. '
        'You take a holistic view — you consider the overall clinical picture, treatment trajectory, '
        'and what the pattern of data implies about disease progression. '
        'You still follow the framework but you also flag clinical nuances.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="deepseek/deepseek-chat"
)

# CONSOLIDATOR: Claude Sonnet
consolidator = Agent(
    role='Chief Medical Informatician — Final Arbiter',
    goal='Review both assessors classifications, resolve any disagreements, and produce a final severity classification with full justification.',
    backstory=(
        'You are the chief medical informatician responsible for the final phenotyping decision. '
        'You review both assessors reasoning, identify where they agree and disagree, '
        'and produce a definitive classification. When assessors disagree, you explain '
        'which evidence you find more compelling and why.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

# ── 4. Define Tasks ───────────────────────────────────────────
print("🔵 Defining Phase 2 Tasks (4 steps)...")

# TASK 1: Extraction (Claude Sonnet — handles large context)
task_extract = Task(
    description=f"""
You are a clinical data extractor. Your job is to read a raw patient EHR record and extract
ALL information relevant to Type 2 Diabetes severity classification, organized by the domains
in the phenotyping framework below.

=== SEVERITY PHENOTYPING FRAMEWORK (for reference — extract data relevant to these domains) ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

=== RAW PATIENT RECORD ===
{raw_patient_record}
=== END RAW PATIENT RECORD ===

INSTRUCTIONS:
1. Read the ENTIRE patient record carefully.
2. For EACH domain in the framework, extract every relevant data point you can find.
3. For observations/labs, include the DATE, VALUE, and UNITS. If multiple values exist over time,
   include ALL of them (the temporal trend matters for assessment).
4. For medications, note which are active (no STOP date) vs stopped.
5. For conditions, note which are active vs resolved.
6. Flag any data that seems anomalous or potentially unreliable (e.g., unrealistic values).
7. If a domain has NO relevant data in the record, explicitly state "No data found for this domain."

OUTPUT FORMAT — organize your extraction under these headings:
- GLYCEMIC DATA: All HbA1c, glucose, fructosamine values with dates
- MEDICATIONS: All diabetes-related medications (oral agents, insulin, GLP-1 etc.) with dates and status
- MICROVASCULAR COMPLICATIONS: Nephropathy, retinopathy, neuropathy, foot ulcer evidence
- MACROVASCULAR COMPLICATIONS: CAD, MI, stroke, PAD evidence
- RENAL FUNCTION: All eGFR, creatinine, UACR/microalbumin values with dates
- COMORBIDITIES: All active chronic conditions (count them)
- FUNCTIONAL STATUS: Any frailty, ADL, cognitive, or BMI data
- TREATMENT INTENSITY: Number of agents, insulin regimen type, device use
- HEALTHCARE UTILIZATION: ED visits, hospitalizations, encounter patterns
- OTHER RELEVANT FINDINGS: Anything else clinically notable
- DATA QUALITY NOTES: Any anomalies, missing data, or concerns
""",
    expected_output='A comprehensive, framework-guided clinical data extraction organized by phenotyping domains, with all relevant values, dates, and data quality notes.',
    agent=extractor
)

# TASK 2: GPT-4o Assessment (async, reads extraction output)
task_assess_gpt = Task(
    description=f"""
You are classifying a Type 2 Diabetes patient's severity using the framework and extracted clinical data below.

=== CONSOLIDATED SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

The clinical data extractor has already processed the raw EHR and organized the relevant findings
for you (provided as context from the previous task).

INSTRUCTIONS:
1. Go through EACH domain/dimension in the framework systematically.
2. For each criterion, state what evidence the extractor found (or did not find).
3. Calculate scores or check thresholds as specified in the framework.
4. Assign a final severity tier (Mild / Moderate / Severe / Critical-Complex).
5. Provide your confidence level (High / Medium / Low) and explain any data gaps.

OUTPUT FORMAT:
- Domain-by-domain assessment with evidence citations
- Summary of which criteria are met vs not met
- Final Classification: [TIER]
- Confidence: [High/Medium/Low]
- Key uncertainties or data gaps
""",
    expected_output='A structured severity assessment with domain-by-domain evidence, final tier classification, and confidence level.',
    agent=assessor_gpt,
    context=[task_extract],
    async_execution=True
)

# TASK 3: DeepSeek Assessment (async, reads extraction output)
task_assess_deepseek = Task(
    description=f"""
You are classifying a Type 2 Diabetes patient's severity using the framework and extracted clinical data below.

=== CONSOLIDATED SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

The clinical data extractor has already processed the raw EHR and organized the relevant findings
for you (provided as context from the previous task).

INSTRUCTIONS:
1. Go through EACH domain/dimension in the framework systematically.
2. For each criterion, state what evidence the extractor found (or did not find).
3. Calculate scores or check thresholds as specified in the framework.
4. Assign a final severity tier (Mild / Moderate / Severe / Critical-Complex).
5. Provide your confidence level (High / Medium / Low) and explain any data gaps.

OUTPUT FORMAT:
- Domain-by-domain assessment with evidence citations
- Summary of which criteria are met vs not met
- Final Classification: [TIER]
- Confidence: [High/Medium/Low]
- Key uncertainties or data gaps
""",
    expected_output='A structured severity assessment with domain-by-domain evidence, final tier classification, and confidence level.',
    agent=assessor_deepseek,
    context=[task_extract],
    async_execution=True
)

# TASK 4: Consolidation (sequential, after both assessors)
task_consolidate = Task(
    description=f"""
You have received severity assessments from two independent clinical assessors for the same T2D patient.

Your job:
1. Compare their domain-by-domain assessments side by side.
2. Identify where they AGREE and where they DISAGREE.
3. For disagreements, examine the evidence and determine which interpretation is more supported.
4. Produce a FINAL consolidated classification.

Patient ID: {PILOT_PATIENT_ID}

OUTPUT FORMAT:
- Agreement Analysis: Where both assessors agree
- Disagreement Analysis: Where they differ, and your resolution
- Final Classification: [TIER]
- Confidence: [High/Medium/Low]
- Evidence Summary: Key factors driving the classification
- Data Quality Notes: Any limitations or artefacts noted
""",
    expected_output='A consolidated severity classification with agreement analysis, final tier, confidence level, and evidence summary.',
    agent=consolidator,
    context=[task_assess_gpt, task_assess_deepseek]
)

# ── 5. Run the Crew ──────────────────────────────────────────
print(f"\n🚀 Phase 2 Kickoff — Classifying Patient {PILOT_PATIENT_ID[:8]}...")
print("   Step 1: Claude Sonnet extracts relevant data (framework-guided)")
print("   Step 2: GPT-4o + DeepSeek assess severity (parallel)")
print("   Step 3: Claude Sonnet consolidates final classification")
print("=" * 60)

crew_phase2 = Crew(
    agents=[extractor, assessor_gpt, assessor_deepseek, consolidator],
    tasks=[task_extract, task_assess_gpt, task_assess_deepseek, task_consolidate],
    verbose=True
)

try:
    phase2_result = crew_phase2.kickoff()
    print("\n✅ Phase 2 Classification Complete!")

    # ── Save all outputs ──────────────────────────────────────
    patient_short = PILOT_PATIENT_ID[:8]

    save_output("phase2/summaries", f"extraction_{patient_short}", str(task_extract.output))
    save_output("phase2/openai", f"assessment_{patient_short}", str(task_assess_gpt.output))
    save_output("phase2/deepseek", f"assessment_{patient_short}", str(task_assess_deepseek.output))
    save_output("phase2/consolidated", f"classification_{patient_short}", str(phase2_result))

    print("\n" + "=" * 60)
    print("📋 FINAL CONSOLIDATED CLASSIFICATION")
    print("=" * 60)
    print(phase2_result)

except Exception as e:
    print(f"❌ Phase 2 failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================================
# 🔧 CELL 6: Extract Clinical Keywords from Consolidated Framework
# Run this ONCE before the 100-patient loop
# Makes the compression disease-agnostic and framework-driven
# (END OF WITH 324 KEYWORDS STILL TOO MUCH INFORMATION TAKEN OUT OF THE PATIENT)
# ============================================================

from crewai import Agent, Task, Crew
import json

print("🔵 Extracting clinical keywords from consolidated framework...")

# Define a simple keyword extraction agent
keyword_agent = Agent(
    role='Clinical Terminology Specialist',
    goal='Extract all clinically relevant keywords and lab test names from a severity phenotyping framework.',
    backstory=(
        'You are a clinical terminology specialist. You read phenotyping frameworks and identify '
        'every lab test, vital sign, biomarker, medication class, and clinical measurement mentioned. '
        'You produce comprehensive keyword lists that can be used to filter EHR observation data.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

keyword_task = Task(
    description=f"""
Read the following severity phenotyping framework and extract ALL keywords that would appear
in EHR observation/lab data. These keywords will be used to filter patient observation records
to keep only clinically relevant measurements.

=== FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

For each keyword, think about how it might appear in an EHR system. Include:
1. Lab test names (e.g., "hemoglobin a1c", "hba1c", "glycated hemoglobin")
2. Vital signs (e.g., "blood pressure", "systolic", "diastolic")
3. Biomarkers (e.g., "egfr", "glomerular", "creatinine")
4. Clinical measurements (e.g., "body mass", "bmi", "weight")
5. Include common synonyms and abbreviations for each

CRITICAL: Output ONLY a valid JSON object with this exact format, nothing else:
{{
    "keywords": ["keyword1", "keyword2", "keyword3", ...],
    "categories": {{
        "glycemic": ["hba1c", "hemoglobin a1c", "glycated", "glucose", "fructosamine", ...],
        "renal": ["egfr", "glomerular", "creatinine", "albumin", "uacr", "microalbumin", ...],
        "lipids": ["cholesterol", "ldl", "hdl", "triglyceride", ...],
        "vitals": ["blood pressure", "systolic", "diastolic", "bmi", "body mass", ...],
        "other": [...]
    }}
}}

Include ALL synonyms and variations. Be comprehensive — missing a keyword means missing patient data.
Output ONLY the JSON, no explanation.
""",
    expected_output='A JSON object containing categorized clinical keywords extracted from the framework.',
    agent=keyword_agent
)

crew_keywords = Crew(
    agents=[keyword_agent],
    tasks=[keyword_task],
    verbose=True
)

keyword_result = crew_keywords.kickoff()

# ── Parse the keywords ────────────────────────────────────────
keyword_text = str(keyword_result)

# Try to parse JSON from the output
try:
    # Clean up common LLM output issues
    if "```json" in keyword_text:
        keyword_text = keyword_text.split("```json")[1].split("```")[0]
    elif "```" in keyword_text:
        keyword_text = keyword_text.split("```")[1].split("```")[0]

    keyword_data = json.loads(keyword_text.strip())

    # Extract the flat keyword list
    T2D_KEYWORDS = [kw.lower() for kw in keyword_data.get("keywords", [])]
    KEYWORD_CATEGORIES = keyword_data.get("categories", {})

    print(f"\n✅ Extracted {len(T2D_KEYWORDS)} clinical keywords from framework")
    print(f"   Categories: {list(KEYWORD_CATEGORIES.keys())}")
    for cat, kws in KEYWORD_CATEGORIES.items():
        print(f"   - {cat}: {', '.join(kws[:5])}{'...' if len(kws) > 5 else ''}")

except (json.JSONDecodeError, IndexError) as e:
    print(f"⚠️ Could not parse JSON from LLM output. Using fallback extraction...")

    # Fallback: extract any quoted strings from the output
    import re
    quoted = re.findall(r'"([^"]+)"', keyword_text)
    # Filter to likely clinical terms (lowercase, reasonable length)
    T2D_KEYWORDS = list(set([q.lower() for q in quoted if 2 < len(q) < 50]))
    KEYWORD_CATEGORIES = {"extracted": T2D_KEYWORDS}

    print(f"✅ Fallback extracted {len(T2D_KEYWORDS)} keywords")

# ── Save keywords for audit trail ─────────────────────────────
save_output("phase2", "framework_keywords", {
    "keywords": T2D_KEYWORDS,
    "categories": KEYWORD_CATEGORIES,
    "source": "extracted from consolidated framework by Claude"
})

print(f"\n📋 Full keyword list ({len(T2D_KEYWORDS)} keywords):")
print(T2D_KEYWORDS)
print("\n✅ Keywords ready — these will be used for patient data compression")
print("   Run the next cell to start the 100-patient classification loop")

In [ ]:
# ============================================================
# 🔧 CELL 7: Extract Clinical Keywords from Consolidated Framework
# Run this ONCE before the 100-patient loop
# Makes the compression disease-agnostic and framework-driven
# (MORE AGRESSIVE APPROACH ENDS UP WITH 22 KEYWORDS FOR TD2)
# ============================================================

# ── FOCUSED T2D Keywords (override the 341 LLM-generated ones) ──
T2D_KEYWORDS = [
    # Glycemic (critical)
    "hba1c", "hemoglobin a1c", "glycated hemoglobin", "glycohemoglobin", "a1c",
    "glucose", "fructosamine",
    # Renal (critical)
    "egfr", "glomerular filtration", "creatinine",
    "microalbumin", "albumin creatinine ratio", "uacr", "urine albumin",
    # Lipids
    "cholesterol", "ldl", "hdl", "triglyceride",
    # Blood pressure
    "systolic blood pressure", "diastolic blood pressure",
    # BMI/Weight
    "body mass index",
    # Kidney extra
    "urea nitrogen",
]

print(f"✅ Focused keyword list: {len(T2D_KEYWORDS)} keywords")

# Verify against worst patient
pid = "082544d6"
full_id = [p for p in ALL_PATIENT_IDS if p.startswith(pid)][0]
obs = observations[observations["PATIENT"] == full_id]
mask = obs["DESCRIPTION"].str.lower().str.contains("|".join(T2D_KEYWORDS), na=False)
unique_matched = obs[mask]["DESCRIPTION"].nunique()
print(f"Worst patient: {mask.sum()} matched / {len(obs)} total ({mask.sum()/len(obs)*100:.0f}%)")
print(f"With last-10 limit: ~{unique_matched * 10} rows max")
print(f"Unmatched (latest only): {obs[~mask]['DESCRIPTION'].nunique()} rows")

In [ ]:
# ============================================================
# 🔧 CELL 8: Phase 2 — APPROACH B PILOT (1 Patient)(VERSION 2)
#
# ARCHITECTURE:
#   Extractor:     GPT-4o (high rate limits, 128K context)
#   Assessor A:    GPT-4o
#   Assessor B:    DeepSeek
#   Consolidator:  Claude Sonnet (only Claude call per patient)
# ============================================================

from crewai import Agent, Task, Crew
import pandas as pd
import time

# ── 1. Setup ──────────────────────────────────────────────────
CONSOLIDATED_FRAMEWORK = str(result)
print(f"✅ Framework: {len(CONSOLIDATED_FRAMEWORK)} chars")
print(f"✅ Keywords: {len(T2D_KEYWORDS)} (focused)")
print(f"🎯 Pilot patient: {PILOT_PATIENT_ID[:8]}")

# ── 2. Compression Function ──────────────────────────────────
MAX_HISTORY_PER_OBS = 10

def compress_patient_data(patient_id):
    sections = {}

    demo = patients[patients["Id"] == patient_id]
    if not demo.empty:
        drop_cols = ["Id", "SSN", "DRIVERS", "PASSPORT", "ADDRESS", "CITY", "STATE",
                     "COUNTY", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE"]
        keep = [c for c in demo.columns if c not in drop_cols]
        sections["demographics"] = demo[keep].to_string(index=False)

    cond = conditions[conditions["PATIENT"] == patient_id]
    if not cond.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in cond.columns if c not in drop_cols]
        sections["conditions"] = cond[keep].to_string(index=False)
    else:
        sections["conditions"] = "No conditions recorded."

    obs = observations[observations["PATIENT"] == patient_id]
    if not obs.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "TYPE"]
        keep = [c for c in obs.columns if c not in drop_cols]
        obs_clean = obs[keep].copy()
        obs_clean["DATE"] = pd.to_datetime(obs_clean["DATE"], errors="coerce")
        obs_clean = obs_clean.sort_values("DATE", ascending=False)
        keyword_pattern = "|".join(T2D_KEYWORDS)
        mask_relevant = obs_clean["DESCRIPTION"].str.lower().str.contains(keyword_pattern, na=False)
        obs_relevant = obs_clean[mask_relevant].groupby("DESCRIPTION").head(MAX_HISTORY_PER_OBS)
        obs_other = obs_clean[~mask_relevant].drop_duplicates(subset=["DESCRIPTION"], keep="first")
        obs_combined = pd.concat([obs_relevant, obs_other]).sort_values("DATE", ascending=False)
        sections["observations"] = (
            f"[{len(obs)} total observations, {len(obs_combined)} after compression]\n"
            + obs_combined.to_string(index=False)
        )
    else:
        sections["observations"] = "No observations recorded."

    meds = medications[medications["PATIENT"] == patient_id]
    if not meds.empty:
        drop_cols = ["PATIENT", "PAYER", "ENCOUNTER", "BASE_COST", "PAYER_COVERAGE", "DISPENSES", "TOTALCOST"]
        keep = [c for c in meds.columns if c not in drop_cols]
        meds_clean = meds[keep].drop_duplicates(subset=["DESCRIPTION", "START", "STOP"])
        sections["medications"] = f"[{len(meds)} total, {len(meds_clean)} unique]\n" + meds_clean.to_string(index=False)
    else:
        sections["medications"] = "No medications recorded."

    enc = encounters[encounters["PATIENT"] == patient_id]
    if not enc.empty:
        enc_summary = enc.groupby("ENCOUNTERCLASS").size().reset_index(name="COUNT")
        enc_with_reason = enc[enc["REASONDESCRIPTION"].notna()]
        if not enc_with_reason.empty:
            enc_reasons = enc_with_reason[["ENCOUNTERCLASS", "REASONDESCRIPTION"]].drop_duplicates()
            sections["encounters"] = (
                f"[{len(enc)} total encounters]\nSUMMARY:\n{enc_summary.to_string(index=False)}\n\n"
                f"CLINICAL REASONS ({len(enc_reasons)}):\n{enc_reasons.to_string(index=False)}"
            )
        else:
            sections["encounters"] = f"[{len(enc)} total]\n{enc_summary.to_string(index=False)}"
    else:
        sections["encounters"] = "No encounters recorded."

    cp = careplans[careplans["PATIENT"] == patient_id]
    if not cp.empty:
        drop_cols = ["Id", "PATIENT", "ENCOUNTER"]
        keep = [c for c in cp.columns if c not in drop_cols]
        sections["careplans"] = cp[keep].to_string(index=False)
    else:
        sections["careplans"] = "No careplans recorded."

    proc = procedures[procedures["PATIENT"] == patient_id]
    if not proc.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "BASE_COST"]
        keep = [c for c in proc.columns if c not in drop_cols]
        proc_clean = proc[keep].sort_values("DATE", ascending=False).drop_duplicates(subset=["DESCRIPTION"], keep="first")
        sections["procedures"] = f"[{len(proc)} total, {len(proc_clean)} unique]\n" + proc_clean.to_string(index=False)
    else:
        sections["procedures"] = "No procedures recorded."

    allg = allergies[allergies["PATIENT"] == patient_id]
    if not allg.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in allg.columns if c not in drop_cols]
        sections["allergies"] = allg[keep].to_string(index=False)
    else:
        sections["allergies"] = "No allergies recorded."

    dev = devices[devices["PATIENT"] == patient_id]
    if not dev.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "UDI"]
        keep = [c for c in dev.columns if c not in drop_cols]
        sections["devices"] = dev[keep].to_string(index=False)
    else:
        sections["devices"] = "No devices recorded."

    sections["immunizations"] = "[Omitted — not relevant]"
    return sections

# ── 3. Compress pilot patient ─────────────────────────────────
print(f"\n🔵 Compressing patient data...")
raw_data = compress_patient_data(PILOT_PATIENT_ID)
raw_patient_record = f"=== PATIENT RECORD: {PILOT_PATIENT_ID} ===\n\n"
for section, content in raw_data.items():
    raw_patient_record += f"--- {section.upper()} ---\n{content}\n\n"

print(f"✅ Patient data: {len(raw_patient_record)} chars (~{len(raw_patient_record)//4} tokens)")
save_output("phase2/summaries", f"patient_{PILOT_PATIENT_ID[:8]}_input", raw_patient_record)

# ── 4. Define Agents (Approach B) ─────────────────────────────
print("\n🔵 Initializing Approach B agents...")

# EXTRACTOR: GPT-4o (NOT Claude — avoids rate limit)
extractor = Agent(
    role='Clinical Data Extractor',
    goal='Extract and organize all clinically relevant information from EHR data, guided by the severity phenotyping framework.',
    backstory=(
        'You are a clinical data specialist who excels at reading electronic health records '
        'and extracting the specific data points needed for clinical phenotyping. '
        'You are thorough — you never miss relevant data — but you also filter out noise. '
        'You organize your output by clinical domain.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# ASSESSOR A: GPT-4o
assessor_gpt = Agent(
    role='Dr. A — Clinical Informatician',
    goal='Classify T2D severity. Be methodical, conservative, evidence-based.',
    backstory=(
        'You are a clinical informatician specializing in EHR-based phenotyping. '
        'You are methodical and conservative — you only count what is explicitly documented. '
        'If data is missing or ambiguous, you note it but do NOT assume severity.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="gpt-4o"
)

# ASSESSOR B: DeepSeek
assessor_deepseek = Agent(
    role='Dr. B — Consultant Endocrinologist',
    goal='Classify T2D severity. Take a holistic, patient-outcomes focused approach.',
    backstory=(
        'You are an endocrinologist with 20 years experience. '
        'You consider the full clinical picture and treatment trajectory.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="deepseek/deepseek-chat"
)

# CONSOLIDATOR: Claude Sonnet (only Claude call — receives short assessments)
consolidator = Agent(
    role='Chief Medical Informatician — Final Arbiter',
    goal='Review both assessors, resolve disagreements, produce final classification.',
    backstory=(
        'You are the chief medical informatician responsible for the final phenotyping decision. '
        'You review both assessors reasoning, identify agreements/disagreements, '
        'and produce a definitive classification with both a four-tier label and a binary label.'
    ),
    verbose=True,
    allow_delegation=False,
    llm="anthropic/claude-sonnet-4-6"
)

# ── 5. Define Tasks ───────────────────────────────────────────
print("🔵 Defining tasks...")

task_extract = Task(
    description=f"""
Extract ALL information relevant to T2D severity classification from this patient record.

=== SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

=== PATIENT RECORD ===
{raw_patient_record}
=== END PATIENT RECORD ===

Organize output under: GLYCEMIC DATA, MEDICATIONS, MICROVASCULAR COMPLICATIONS,
MACROVASCULAR COMPLICATIONS, RENAL FUNCTION, COMORBIDITIES, FUNCTIONAL STATUS,
TREATMENT INTENSITY, HEALTHCARE UTILIZATION, OTHER FINDINGS, DATA QUALITY NOTES.
""",
    expected_output='Framework-guided clinical extraction by domain.',
    agent=extractor
)

assess_prompt = f"""
Classify this T2D patient using the framework and extracted data (from context).

=== SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

Go through EACH domain. Assign BOTH:
a) Four-tier: Mild / Moderate / Severe / Critical-Complex
b) Binary: SEVERE or NOT_SEVERE (Severe+Critical=SEVERE, Mild+Moderate=NOT_SEVERE)
Confidence: High / Medium / Low.
"""

task_assess_gpt = Task(
    description=assess_prompt,
    expected_output='Severity assessment with four-tier and binary classification.',
    agent=assessor_gpt,
    context=[task_extract],
    async_execution=True
)

task_assess_deepseek = Task(
    description=assess_prompt,
    expected_output='Severity assessment with four-tier and binary classification.',
    agent=assessor_deepseek,
    context=[task_extract],
    async_execution=True
)

task_consolidate = Task(
    description=f"""
Review both assessors' assessments and produce a final classification.
Patient ID: {PILOT_PATIENT_ID}

CRITICAL — End with this exact format:

===FINAL===
PATIENT_ID: {PILOT_PATIENT_ID}
FOUR_TIER: [Mild/Moderate/Severe/Critical-Complex]
BINARY: [SEVERE/NOT_SEVERE]
ASSESSOR_A_TIER: [Dr. A's tier]
ASSESSOR_B_TIER: [Dr. B's tier]
ASSESSOR_A_BINARY: [Dr. A's binary]
ASSESSOR_B_BINARY: [Dr. B's binary]
CONFIDENCE: [High/Medium/Low]
AGREEMENT: [Full/Partial/None]
===END===

Before that: Agreement Analysis, Disagreement Analysis, Evidence Summary, Data Quality Notes.
""",
    expected_output='Consolidated classification with ===FINAL=== block.',
    agent=consolidator,
    context=[task_assess_gpt, task_assess_deepseek]
)

# ── 6. Run ────────────────────────────────────────────────────
print(f"\n🚀 Approach B Pilot — Patient {PILOT_PATIENT_ID[:8]}")
print("   Extractor: GPT-4o | Assessors: GPT-4o + DeepSeek | Consolidator: Claude")
print("=" * 60)

crew_pilot = Crew(
    agents=[extractor, assessor_gpt, assessor_deepseek, consolidator],
    tasks=[task_extract, task_assess_gpt, task_assess_deepseek, task_consolidate],
    verbose=True
)

start = time.time()

try:
    pilot_result = crew_pilot.kickoff()
    elapsed = time.time() - start

    print(f"\n✅ Pilot complete in {elapsed:.0f}s")

    # Save
    patient_short = PILOT_PATIENT_ID[:8]
    save_output("phase2/summaries", f"extraction_{patient_short}", str(task_extract.output))
    save_output("phase2/openai", f"assessment_{patient_short}", str(task_assess_gpt.output))
    save_output("phase2/deepseek", f"assessment_{patient_short}", str(task_assess_deepseek.output))
    save_output("phase2/consolidated", f"classification_{patient_short}", str(pilot_result))

    # Parse
    result_text = str(pilot_result)
    if "===FINAL===" in result_text:
        final_block = result_text.split("===FINAL===")[1].split("===END===")[0]
        print("\n📋 PARSED RESULT:")
        print(final_block.strip())

    print("\n" + "=" * 60)
    print("📋 FULL CONSOLIDATED OUTPUT:")
    print("=" * 60)
    print(pilot_result)

except Exception as e:
    print(f"❌ Failed: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# ============================================================
# 🔧 CELL 9: Phase 2 — APPROACH B — 20-PATIENT TEST
#
# ARCHITECTURE:
#   Extractor:     GPT-4o (high rate limits, 128K context)
#   Assessor A:    GPT-4o
#   Assessor B:    DeepSeek
#   Consolidator:  Claude Sonnet (only Claude call per patient)
#
# Rate limit safe: Claude only sees ~5K tokens per patient
# ============================================================

from crewai import Agent, Task, Crew
import pandas as pd
import time
import json
import os

# ── 1. Setup ──────────────────────────────────────────────────
CONSOLIDATED_FRAMEWORK = str(result)
print(f"✅ Framework: {len(CONSOLIDATED_FRAMEWORK)} chars")
print(f"✅ Keywords: {len(T2D_KEYWORDS)} (focused)")

TEST_PATIENT_IDS = patient_list["PATIENT_ID"].tolist()[:20]
print(f"✅ Patients: {len(TEST_PATIENT_IDS)}")

RESULTS_DIR = f"{OUTPUT_DIR}/phase2"
MAX_HISTORY_PER_OBS = 10
COOLDOWN_SECONDS = 30  # Only 30s needed — Claude input is tiny now

# ── 2. Checkpoint ─────────────────────────────────────────────
def get_completed_patients():
    completed = set()
    consol_dir = f"{RESULTS_DIR}/consolidated"
    if os.path.exists(consol_dir):
        for f in os.listdir(consol_dir):
            if f.startswith("classification_") and f.endswith(".json"):
                completed.add(f.replace("classification_", "").replace(".json", ""))
    return completed

completed = get_completed_patients()
print(f"✅ Checkpoint: {len(completed)} already done")

# ── 3. Compression Function ──────────────────────────────────
def compress_patient_data(patient_id):
    sections = {}

    demo = patients[patients["Id"] == patient_id]
    if not demo.empty:
        drop_cols = ["Id", "SSN", "DRIVERS", "PASSPORT", "ADDRESS", "CITY", "STATE",
                     "COUNTY", "ZIP", "LAT", "LON", "HEALTHCARE_EXPENSES", "HEALTHCARE_COVERAGE"]
        keep = [c for c in demo.columns if c not in drop_cols]
        sections["demographics"] = demo[keep].to_string(index=False)

    cond = conditions[conditions["PATIENT"] == patient_id]
    if not cond.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in cond.columns if c not in drop_cols]
        sections["conditions"] = cond[keep].to_string(index=False)
    else:
        sections["conditions"] = "No conditions recorded."

    obs = observations[observations["PATIENT"] == patient_id]
    if not obs.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "TYPE"]
        keep = [c for c in obs.columns if c not in drop_cols]
        obs_clean = obs[keep].copy()
        obs_clean["DATE"] = pd.to_datetime(obs_clean["DATE"], errors="coerce")
        obs_clean = obs_clean.sort_values("DATE", ascending=False)
        keyword_pattern = "|".join(T2D_KEYWORDS)
        mask_relevant = obs_clean["DESCRIPTION"].str.lower().str.contains(keyword_pattern, na=False)
        obs_relevant = obs_clean[mask_relevant].groupby("DESCRIPTION").head(MAX_HISTORY_PER_OBS)
        obs_other = obs_clean[~mask_relevant].drop_duplicates(subset=["DESCRIPTION"], keep="first")
        obs_combined = pd.concat([obs_relevant, obs_other]).sort_values("DATE", ascending=False)
        sections["observations"] = (
            f"[{len(obs)} total observations, {len(obs_combined)} after compression]\n"
            + obs_combined.to_string(index=False)
        )
    else:
        sections["observations"] = "No observations recorded."

    meds = medications[medications["PATIENT"] == patient_id]
    if not meds.empty:
        drop_cols = ["PATIENT", "PAYER", "ENCOUNTER", "BASE_COST", "PAYER_COVERAGE", "DISPENSES", "TOTALCOST"]
        keep = [c for c in meds.columns if c not in drop_cols]
        meds_clean = meds[keep].drop_duplicates(subset=["DESCRIPTION", "START", "STOP"])
        sections["medications"] = f"[{len(meds)} total, {len(meds_clean)} unique]\n" + meds_clean.to_string(index=False)
    else:
        sections["medications"] = "No medications recorded."

    enc = encounters[encounters["PATIENT"] == patient_id]
    if not enc.empty:
        enc_summary = enc.groupby("ENCOUNTERCLASS").size().reset_index(name="COUNT")
        enc_with_reason = enc[enc["REASONDESCRIPTION"].notna()]
        if not enc_with_reason.empty:
            enc_reasons = enc_with_reason[["ENCOUNTERCLASS", "REASONDESCRIPTION"]].drop_duplicates()
            sections["encounters"] = (
                f"[{len(enc)} total encounters]\nSUMMARY:\n{enc_summary.to_string(index=False)}\n\n"
                f"CLINICAL REASONS ({len(enc_reasons)}):\n{enc_reasons.to_string(index=False)}"
            )
        else:
            sections["encounters"] = f"[{len(enc)} total]\n{enc_summary.to_string(index=False)}"
    else:
        sections["encounters"] = "No encounters recorded."

    cp = careplans[careplans["PATIENT"] == patient_id]
    if not cp.empty:
        drop_cols = ["Id", "PATIENT", "ENCOUNTER"]
        keep = [c for c in cp.columns if c not in drop_cols]
        sections["careplans"] = cp[keep].to_string(index=False)
    else:
        sections["careplans"] = "No careplans recorded."

    proc = procedures[procedures["PATIENT"] == patient_id]
    if not proc.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "BASE_COST"]
        keep = [c for c in proc.columns if c not in drop_cols]
        proc_clean = proc[keep].sort_values("DATE", ascending=False).drop_duplicates(subset=["DESCRIPTION"], keep="first")
        sections["procedures"] = f"[{len(proc)} total, {len(proc_clean)} unique]\n" + proc_clean.to_string(index=False)
    else:
        sections["procedures"] = "No procedures recorded."

    allg = allergies[allergies["PATIENT"] == patient_id]
    if not allg.empty:
        drop_cols = ["PATIENT", "ENCOUNTER"]
        keep = [c for c in allg.columns if c not in drop_cols]
        sections["allergies"] = allg[keep].to_string(index=False)
    else:
        sections["allergies"] = "No allergies recorded."

    dev = devices[devices["PATIENT"] == patient_id]
    if not dev.empty:
        drop_cols = ["PATIENT", "ENCOUNTER", "UDI"]
        keep = [c for c in dev.columns if c not in drop_cols]
        sections["devices"] = dev[keep].to_string(index=False)
    else:
        sections["devices"] = "No devices recorded."

    sections["immunizations"] = "[Omitted — not relevant]"
    return sections

# ── 4. Pre-flight ─────────────────────────────────────────────
print("\n🔵 Pre-flight check (GPT-4o limit: ~512K chars)...")
all_fit = True
for pid in TEST_PATIENT_IDS:
    data = compress_patient_data(pid)
    size = sum(len(str(v)) for v in data.values())
    status = "✅" if size < 400000 else "⚠️"
    if size >= 400000:
        all_fit = False
    print(f"   {status} {pid[:8]}: {size} chars")

if all_fit:
    print(f"\n✅ All 20 patients fit GPT-4o context window!")
else:
    print(f"\n⚠️ Some patients large — emergency compression will apply if needed.")

# ── 5. Define Agents (ONCE — Approach B) ──────────────────────
print("\n🔵 Initializing Approach B agents...")
print("   Extractor: GPT-4o | Assessors: GPT-4o + DeepSeek | Consolidator: Claude")

extractor = Agent(
    role='Clinical Data Extractor',
    goal='Extract and organize all clinically relevant information from EHR data, guided by the severity phenotyping framework.',
    backstory='You are a clinical data specialist who reads EHR records and extracts data points needed for clinical phenotyping. You organize output by clinical domain.',
    verbose=True, allow_delegation=False, llm="gpt-4o"
)

assessor_gpt = Agent(
    role='Dr. A — Clinical Informatician',
    goal='Classify T2D severity. Be methodical, conservative, evidence-based.',
    backstory='You are a clinical informatician. You only count what is explicitly documented. Missing data is noted but not assumed.',
    verbose=True, allow_delegation=False, llm="gpt-4o"
)

assessor_deepseek = Agent(
    role='Dr. B — Consultant Endocrinologist',
    goal='Classify T2D severity. Take a holistic, patient-outcomes focused approach.',
    backstory='You are an endocrinologist with 20 years experience. You consider the full clinical picture and treatment trajectory.',
    verbose=True, allow_delegation=False, llm="deepseek/deepseek-chat"
)

consolidator = Agent(
    role='Chief Medical Informatician — Final Arbiter',
    goal='Review both assessors, resolve disagreements, produce final classification.',
    backstory='You are the chief medical informatician. You review both assessors reasoning, resolve disagreements, and produce a definitive classification.',
    verbose=True, allow_delegation=False, llm="anthropic/claude-sonnet-4-6"
)

# ── 6. Process Loop ──────────────────────────────────────────
print(f"\n🚀 Phase 2 Approach B — {len(TEST_PATIENT_IDS)} patients")
print(f"   Cooldown: {COOLDOWN_SECONDS}s between patients")
print(f"   Estimated time: ~{len(TEST_PATIENT_IDS) * 3:.0f} minutes")
print("=" * 60)

results_summary = []
errors = []
start_time = time.time()

for i, patient_id in enumerate(TEST_PATIENT_IDS):
    patient_short = patient_id[:8]

    if patient_short in completed:
        print(f"\n⏭️  [{i+1}/20] {patient_short} — skipping (done)")
        continue

    print(f"\n{'='*60}")
    print(f"🔵 [{i+1}/20] Patient: {patient_short}")
    print(f"{'='*60}")
    patient_start = time.time()

    try:
        # Compress
        raw_data = compress_patient_data(patient_id)
        raw_patient_record = f"=== PATIENT RECORD: {patient_id} ===\n\n"
        for section, content in raw_data.items():
            raw_patient_record += f"--- {section.upper()} ---\n{content}\n\n"

        record_chars = len(raw_patient_record)
        print(f"   📊 {record_chars} chars (~{record_chars//4} tokens)")

        # Emergency compression if too large for GPT-4o (128K tokens ≈ 512K chars)
        if record_chars > 400000:
            print(f"   ⚠️ Emergency: reducing to last 3 per obs type...")
            obs = observations[observations["PATIENT"] == patient_id]
            if not obs.empty:
                drop_cols = ["PATIENT", "ENCOUNTER", "TYPE"]
                keep = [c for c in obs.columns if c not in drop_cols]
                obs_clean = obs[keep].copy()
                obs_clean["DATE"] = pd.to_datetime(obs_clean["DATE"], errors="coerce")
                obs_clean = obs_clean.sort_values("DATE", ascending=False)
                keyword_pattern = "|".join(T2D_KEYWORDS)
                mask = obs_clean["DESCRIPTION"].str.lower().str.contains(keyword_pattern, na=False)
                obs_relevant = obs_clean[mask].groupby("DESCRIPTION").head(3)
                obs_other = obs_clean[~mask].drop_duplicates(subset=["DESCRIPTION"], keep="first")
                obs_final = pd.concat([obs_relevant, obs_other]).sort_values("DATE", ascending=False)
                raw_data["observations"] = f"[EMERGENCY: {len(obs_final)} rows]\n" + obs_final.to_string(index=False)

            raw_patient_record = f"=== PATIENT RECORD: {patient_id} ===\n\n"
            for section, content in raw_data.items():
                raw_patient_record += f"--- {section.upper()} ---\n{content}\n\n"
            record_chars = len(raw_patient_record)
            print(f"   📊 After emergency: {record_chars} chars")

            if record_chars > 400000:
                print(f"   ❌ Skipping — still too large")
                errors.append({"patient_id": patient_id, "error": f"Too large: {record_chars}"})
                continue

        save_output("phase2/summaries", f"patient_{patient_short}_input", raw_patient_record)

        # Tasks
        task_extract = Task(
            description=f"""
Extract ALL information relevant to T2D severity classification from this patient record.

=== SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

=== PATIENT RECORD ===
{raw_patient_record}
=== END PATIENT RECORD ===

Organize output under: GLYCEMIC DATA, MEDICATIONS, MICROVASCULAR COMPLICATIONS,
MACROVASCULAR COMPLICATIONS, RENAL FUNCTION, COMORBIDITIES, FUNCTIONAL STATUS,
TREATMENT INTENSITY, HEALTHCARE UTILIZATION, OTHER FINDINGS, DATA QUALITY NOTES.
""",
            expected_output='Framework-guided clinical extraction by domain.',
            agent=extractor
        )

        assess_prompt = f"""
Classify this T2D patient using the framework and extracted data (from context).

=== SEVERITY PHENOTYPING FRAMEWORK ===
{CONSOLIDATED_FRAMEWORK}
=== END FRAMEWORK ===

Go through EACH domain. Assign BOTH:
a) Four-tier: Mild / Moderate / Severe / Critical-Complex
b) Binary: SEVERE or NOT_SEVERE (Severe+Critical=SEVERE, Mild+Moderate=NOT_SEVERE)
Confidence: High / Medium / Low.
"""

        task_assess_gpt = Task(
            description=assess_prompt,
            expected_output='Severity assessment with four-tier and binary classification.',
            agent=assessor_gpt, context=[task_extract], async_execution=True
        )

        task_assess_deepseek = Task(
            description=assess_prompt,
            expected_output='Severity assessment with four-tier and binary classification.',
            agent=assessor_deepseek, context=[task_extract], async_execution=True
        )

        task_consolidate = Task(
            description=f"""
Review both assessors' assessments and produce a final classification.
Patient ID: {patient_id}

CRITICAL — End with this exact format:

===FINAL===
PATIENT_ID: {patient_id}
FOUR_TIER: [Mild/Moderate/Severe/Critical-Complex]
BINARY: [SEVERE/NOT_SEVERE]
ASSESSOR_A_TIER: [Dr. A's tier]
ASSESSOR_B_TIER: [Dr. B's tier]
ASSESSOR_A_BINARY: [Dr. A's binary]
ASSESSOR_B_BINARY: [Dr. B's binary]
CONFIDENCE: [High/Medium/Low]
AGREEMENT: [Full/Partial/None]
===END===

Before that: Agreement Analysis, Disagreement Analysis, Evidence Summary, Data Quality Notes.
""",
            expected_output='Consolidated classification with ===FINAL=== block.',
            agent=consolidator, context=[task_assess_gpt, task_assess_deepseek]
        )

        # Run
        crew_patient = Crew(
            agents=[extractor, assessor_gpt, assessor_deepseek, consolidator],
            tasks=[task_extract, task_assess_gpt, task_assess_deepseek, task_consolidate],
            verbose=True
        )

        patient_result = crew_patient.kickoff()

        # Save
        save_output("phase2/summaries", f"extraction_{patient_short}", str(task_extract.output))
        save_output("phase2/openai", f"assessment_{patient_short}", str(task_assess_gpt.output))
        save_output("phase2/deepseek", f"assessment_{patient_short}", str(task_assess_deepseek.output))
        save_output("phase2/consolidated", f"classification_{patient_short}", str(patient_result))

        # Parse
        result_text = str(patient_result)
        parsed = {"patient_id": patient_id, "patient_short": patient_short}

        if "===FINAL===" in result_text:
            final_block = result_text.split("===FINAL===")[1].split("===END===")[0]
            for line in final_block.strip().split("\n"):
                if ":" in line:
                    key, val = line.split(":", 1)
                    parsed[key.strip()] = val.strip()
            print(f"   ✅ {parsed.get('FOUR_TIER', '?')} / {parsed.get('BINARY', '?')} "
                  f"(Conf: {parsed.get('CONFIDENCE', '?')}, Agree: {parsed.get('AGREEMENT', '?')})")
        else:
            parsed["PARSE_ERROR"] = "No ===FINAL=== block"
            print(f"   ⚠️ Could not parse — saved raw")

        results_summary.append(parsed)

        # Timing
        elapsed = time.time() - patient_start
        done = len(completed) + len(results_summary)
        avg = (time.time() - start_time) / len(results_summary)
        eta = avg * (20 - done)
        print(f"   ⏱️  {elapsed:.0f}s | Done: {done}/20 | ETA: {eta/60:.0f}min")

    except Exception as e:
        print(f"   ❌ FAILED: {e}")
        errors.append({"patient_id": patient_id, "error": str(e)})
        with open(f"{RESULTS_DIR}/errors.json", "w") as f:
            json.dump(errors, f, indent=2)

    # Cooldown between patients
    if i < len(TEST_PATIENT_IDS) - 1:
        print(f"   ⏳ Cooldown {COOLDOWN_SECONDS}s...")
        time.sleep(COOLDOWN_SECONDS)

# ── 7. Summary ────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🏁 20-PATIENT TEST COMPLETE")
print("=" * 60)
print(f"⏱️  Total: {(time.time()-start_time)/60:.1f} min")
print(f"✅ Completed: {len(results_summary)}")
print(f"⏭️  Skipped: {len(completed)}")
print(f"❌ Errors: {len(errors)}")

if errors:
    for err in errors:
        print(f"   {err['patient_id'][:8]}: {err['error'][:80]}")

if results_summary:
    df = pd.DataFrame(results_summary)
    df.to_csv(f"{RESULTS_DIR}/results_summary_20.csv", index=False)

    print("\n--- Distribution ---")
    if "FOUR_TIER" in df.columns:
        print(f"\nFour-Tier:\n{df['FOUR_TIER'].value_counts().to_string()}")
    if "BINARY" in df.columns:
        print(f"\nBinary:\n{df['BINARY'].value_counts().to_string()}")
    if "AGREEMENT" in df.columns:
        print(f"\nAgreement:\n{df['AGREEMENT'].value_counts().to_string()}")

import shutil
from google.colab import files
shutil.make_archive("/content/phase2_test_20", "zip", RESULTS_DIR)
files.download("/content/phase2_test_20.zip")
print("\n📥 Downloading!")

In [ ]:
# ============================================================
# 🔧 CELL 10: OVERVIEW OF RESULTS
# ============================================================
import os
import pandas as pd

RESULTS_DIR = f"{OUTPUT_DIR}/phase2"
consol_dir = f"{RESULTS_DIR}/consolidated"

# Load all .txt classification files
results = []
for f in sorted(os.listdir(consol_dir)):
    if not f.endswith(".txt"):
        continue

    filepath = os.path.join(consol_dir, f)
    with open(filepath, "r") as fh:
        text = fh.read()

    patient_short = f.replace("classification_", "").replace(".txt", "")
    parsed = {"patient_short": patient_short, "full_text": text}

    if "===FINAL===" in text:
        try:
            final_block = text.split("===FINAL===")[1].split("===END===")[0]
            for line in final_block.strip().split("\n"):
                if ":" in line:
                    key, val = line.split(":", 1)
                    parsed[key.strip()] = val.strip()
        except:
            parsed["PARSE_ERROR"] = "Failed to parse"
    else:
        parsed["PARSE_ERROR"] = "No ===FINAL=== block"

    results.append(parsed)

df = pd.DataFrame(results)
print(f"✅ Loaded {len(df)} completed classifications\n")

# ── Distributions ─────────────────────────────────────────────
print("=" * 60)
print("📊 CLASSIFICATION RESULTS")
print("=" * 60)

if "FOUR_TIER" in df.columns:
    print("\n🏷️  FOUR-TIER:")
    for tier, count in df["FOUR_TIER"].value_counts().items():
        print(f"   {tier:<25} {count:>3}  {'█' * (count * 3)}")

if "BINARY" in df.columns:
    print("\n🏷️  BINARY:")
    for label, count in df["BINARY"].value_counts().items():
        print(f"   {label:<25} {count:>3}  {'█' * (count * 3)}")

if "AGREEMENT" in df.columns:
    print("\n🤝 AGREEMENT:")
    for level, count in df["AGREEMENT"].value_counts().items():
        print(f"   {level:<25} {count:>3}  {'█' * (count * 3)}")

if "CONFIDENCE" in df.columns:
    print("\n🎯 CONFIDENCE:")
    for level, count in df["CONFIDENCE"].value_counts().items():
        print(f"   {level:<25} {count:>3}  {'█' * (count * 3)}")

# ── Comparison Table ──────────────────────────────────────────
print("\n" + "=" * 60)
print("🔍 ASSESSOR COMPARISON")
print("=" * 60)
print(f"\n{'Patient':<12} {'Dr.A (GPT-4o)':<22} {'Dr.B (DeepSeek)':<22} {'Final':<22} {'Agree'}")
print("-" * 88)
for _, row in df.iterrows():
    print(f"{row['patient_short']:<12} {row.get('ASSESSOR_A_TIER','?'):<22} {row.get('ASSESSOR_B_TIER','?'):<22} {row.get('FOUR_TIER','?'):<22} {row.get('AGREEMENT','?')}")

print(f"\n{'Patient':<12} {'Dr.A Binary':<18} {'Dr.B Binary':<18} {'Final Binary'}")
print("-" * 60)
for _, row in df.iterrows():
    print(f"{row['patient_short']:<12} {row.get('ASSESSOR_A_BINARY','?'):<18} {row.get('ASSESSOR_B_BINARY','?'):<18} {row.get('BINARY','?')}")

# ── Evidence Per Patient ──────────────────────────────────────
print("\n" + "=" * 60)
print("📋 EVIDENCE PER PATIENT")
print("=" * 60)

for _, row in df.iterrows():
    print(f"\n{'─'*60}")
    print(f"🔹 {row['patient_short']} → {row.get('FOUR_TIER','?')} ({row.get('BINARY','?')}) | Conf: {row.get('CONFIDENCE','?')} | Agree: {row.get('AGREEMENT','?')}")
    print(f"{'─'*60}")

    text = row.get("full_text", "")
    # Show the reasoning before ===FINAL=== (that's the evidence)
    if "===FINAL===" in text:
        reasoning = text.split("===FINAL===")[0].strip()
    else:
        reasoning = text[:2000]

    # Show up to 1500 chars of reasoning
    if len(reasoning) > 1500:
        reasoning = reasoning[:1500] + "\n   ... [truncated]"

    for line in reasoning.split("\n"):
        print(f"   {line}")

# ── Summary Stats ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("📈 SUMMARY")
print("=" * 60)
total = len(df)
if "BINARY" in df.columns:
    severe = (df["BINARY"] == "SEVERE").sum()
    not_severe = (df["BINARY"] == "NOT_SEVERE").sum()
    print(f"   Classified: {total} | SEVERE: {severe} ({severe/total*100:.0f}%) | NOT_SEVERE: {not_severe} ({not_severe/total*100:.0f}%)")
if "AGREEMENT" in df.columns:
    full = (df["AGREEMENT"] == "Full").sum()
    print(f"   Full agreement: {full}/{total} ({full/total*100:.0f}%)")

In [ ]:
# ============================================================
# 🔧 CELL 11: VIEW ONE PATIENT LOG
# ============================================================
# Change this to any patient_short ID you want to inspect
PATIENT_TO_VIEW = "5a1a39df"

import os

RESULTS_DIR = f"{OUTPUT_DIR}/phase2"

print(f"📋 FULL LOG FOR PATIENT: {PATIENT_TO_VIEW}")
print("=" * 60)

# 1. Input data sent to extractor
input_file = f"{RESULTS_DIR}/summaries/patient_{PATIENT_TO_VIEW}_input.txt"
if os.path.exists(input_file):
    with open(input_file, "r") as f:
        print(f"\n{'='*60}")
        print(f"📥 RAW INPUT DATA (compressed)")
        print(f"{'='*60}")
        print(f.read())

# 2. Extractor output
extract_file = f"{RESULTS_DIR}/summaries/extraction_{PATIENT_TO_VIEW}.txt"
if os.path.exists(extract_file):
    with open(extract_file, "r") as f:
        print(f"\n{'='*60}")
        print(f"🔍 EXTRACTOR OUTPUT (GPT-4o)")
        print(f"{'='*60}")
        print(f.read())

# 3. Dr. A assessment
gpt_file = f"{RESULTS_DIR}/openai/assessment_{PATIENT_TO_VIEW}.txt"
if os.path.exists(gpt_file):
    with open(gpt_file, "r") as f:
        print(f"\n{'='*60}")
        print(f"🅰️  DR. A — GPT-4o ASSESSMENT")
        print(f"{'='*60}")
        print(f.read())

# 4. Dr. B assessment
ds_file = f"{RESULTS_DIR}/deepseek/assessment_{PATIENT_TO_VIEW}.txt"
if os.path.exists(ds_file):
    with open(ds_file, "r") as f:
        print(f"\n{'='*60}")
        print(f"🅱️  DR. B — DeepSeek ASSESSMENT")
        print(f"{'='*60}")
        print(f.read())

# 5. Consolidated classification
consol_file = f"{RESULTS_DIR}/consolidated/classification_{PATIENT_TO_VIEW}.txt"
if os.path.exists(consol_file):
    with open(consol_file, "r") as f:
        print(f"\n{'='*60}")
        print(f"⚖️  CONSOLIDATED FINAL CLASSIFICATION (Claude)")
        print(f"{'='*60}")
        print(f.read())

print(f"\n{'='*60}")
print(f"✅ Full log complete for patient {PATIENT_TO_VIEW}")

In [ ]:
# ============================================================
# 🔧 CELL 12: UPLOAD GROUND TRUTH FILES
# ============================================================

# ── Upload Ground Truth File ──────────────────────────────────
from google.colab import files

print("📤 Please upload the ground truth file...")
uploaded = files.upload()

GT_FILE = list(uploaded.keys())[0]
gt_df = pd.read_excel(GT_FILE)

print(f"\n✅ Ground truth loaded: {len(gt_df)} patients")
print(f"   Columns: {list(gt_df.columns)}")
print(f"   Label distribution:")
for col in gt_df.columns:
    if gt_df[col].nunique() < 10:
        print(f"   - {col}: {dict(gt_df[col].value_counts())}")
print(f"\n   First 5 rows:")
print(gt_df.head())

In [ ]:
# ============================================================
# 🔧 CELL 13: Phase 3 — EVALUATION vs GROUND TRUTH
#
# ⚠️ UNSEALING GROUND TRUTH
# The pipeline has already classified patients independently.
# Ground truth was sealed before any pipeline runs.
# This comparison does NOT create circular reasoning.
#
# Metrics: Accuracy, Sensitivity, Specificity, Cohen's Kappa,
#          Confusion Matrix, Inter-Assessor Agreement
# ============================================================

import os
import pandas as pd
import numpy as np

RESULTS_DIR = f"{OUTPUT_DIR}/phase2"
consol_dir = f"{RESULTS_DIR}/consolidated"

# ── 1. Load Pipeline Results ──────────────────────────────────
print("🔵 Loading pipeline classifications...")

results = []
for f in sorted(os.listdir(consol_dir)):
    if not f.endswith(".txt"):
        continue

    filepath = os.path.join(consol_dir, f)
    with open(filepath, "r") as fh:
        text = fh.read()

    patient_short = f.replace("classification_", "").replace(".txt", "")
    parsed = {"patient_short": patient_short}

    if "===FINAL===" in text:
        try:
            final_block = text.split("===FINAL===")[1].split("===END===")[0]
            for line in final_block.strip().split("\n"):
                if ":" in line:
                    key, val = line.split(":", 1)
                    parsed[key.strip()] = val.strip()
        except:
            parsed["PARSE_ERROR"] = "Failed to parse"
    else:
        parsed["PARSE_ERROR"] = "No ===FINAL=== block"

    results.append(parsed)

pipeline_df = pd.DataFrame(results)
print(f"✅ Loaded {len(pipeline_df)} pipeline classifications")

# ── 2. Load Ground Truth ─────────────────────────────────────
print("\n🔓 UNSEALING GROUND TRUTH...")

# UPDATE THIS PATH to your ground truth file location
GROUND_TRUTH_PATH = "/content/t2d_100_patients_groundtruth_SEALED.xlsx"

# Try loading — adjust path if needed
try:
    gt_df = pd.read_excel(GROUND_TRUTH_PATH)
    print(f"✅ Ground truth loaded: {len(gt_df)} patients")
    print(f"   Columns: {list(gt_df.columns)}")
    print(f"   First few rows:")
    print(gt_df.head())
except FileNotFoundError:
    print(f"❌ File not found at: {GROUND_TRUTH_PATH}")
    print("   Please update GROUND_TRUTH_PATH to the correct location.")
    print("   Then re-run this cell.")
    gt_df = None

if gt_df is not None:
    # ── 3. Identify Ground Truth Columns ──────────────────────
    # Print columns so we can identify the right ones
    print(f"\n📋 Ground truth columns: {list(gt_df.columns)}")
    print(f"   Unique values in each column:")
    for col in gt_df.columns:
        unique = gt_df[col].nunique()
        if unique < 10:
            print(f"   - {col}: {gt_df[col].unique()}")

    # ── 4. Match Pipeline Results with Ground Truth ───────────
    # The ground truth likely has full patient IDs
    # Our pipeline results have short IDs (first 8 chars)
    # We need to match them

    # Try to find the patient ID column in ground truth
    id_col = None
    binary_col = None

    for col in gt_df.columns:
        col_lower = col.lower().replace(" ", "_")
        if "patient" in col_lower and "id" in col_lower:
            id_col = col
        elif col_lower in ["severity", "binary", "label", "class", "severe"]:
            binary_col = col
        elif col_lower in ["ground_truth", "groundtruth", "gt", "truth"]:
            binary_col = col

    id_col = "PATIENT_ID"
    binary_col = "RULE_LABEL"

    if id_col is None:
        print("\n⚠️ Could not auto-detect patient ID column.")
        print(f"   Available columns: {list(gt_df.columns)}")
        print("   Please set id_col manually below and re-run.")
        # id_col = "YOUR_COLUMN_NAME"

    if binary_col is None:
        print("\n⚠️ Could not auto-detect severity label column.")
        print(f"   Available columns: {list(gt_df.columns)}")
        print("   Please set binary_col manually below and re-run.")
        # binary_col = "YOUR_COLUMN_NAME"

    if id_col and binary_col:
        print(f"\n✅ Using: ID column = '{id_col}', Label column = '{binary_col}'")

        # Create short IDs for ground truth matching
        gt_df["patient_short"] = gt_df[id_col].astype(str).str[:8]

        # Merge
        merged = pipeline_df.merge(gt_df[["patient_short", binary_col]], on="patient_short", how="inner")
        merged = merged.rename(columns={binary_col: "GROUND_TRUTH"})

        # Standardize labels
        merged["GROUND_TRUTH"] = merged["GROUND_TRUTH"].str.strip().str.upper()
        merged["PIPELINE_BINARY"] = merged["BINARY"].str.strip().str.upper()

        print(f"\n✅ Matched {len(merged)} patients (pipeline ∩ ground truth)")
        print(f"   Pipeline total: {len(pipeline_df)}")
        print(f"   Ground truth total: {len(gt_df)}")

        if len(merged) == 0:
            print("❌ No matches found! Check ID format.")
        else:
            # ── 5. Confusion Matrix ───────────────────────────
            print("\n" + "=" * 60)
            print("📊 EVALUATION RESULTS")
            print("=" * 60)

            # Manual confusion matrix
            tp = ((merged["PIPELINE_BINARY"] == "SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()
            tn = ((merged["PIPELINE_BINARY"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
            fp = ((merged["PIPELINE_BINARY"] == "SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
            fn = ((merged["PIPELINE_BINARY"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()

            n = len(merged)

            print(f"\n🔲 CONFUSION MATRIX (n={n})")
            print(f"{'':>25} {'Predicted SEVERE':>18} {'Predicted NOT_SEVERE':>22}")
            print(f"   {'Actual SEVERE':<20} {tp:>18} {fn:>22}")
            print(f"   {'Actual NOT_SEVERE':<20} {fp:>18} {tn:>22}")

            # ── 6. Metrics ────────────────────────────────────
            accuracy = (tp + tn) / n if n > 0 else 0
            sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall for SEVERE
            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0  # Recall for NOT_SEVERE
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            f1 = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0

            # Cohen's Kappa
            p_observed = accuracy
            p_severe_pred = (tp + fp) / n if n > 0 else 0
            p_severe_actual = (tp + fn) / n if n > 0 else 0
            p_not_pred = (tn + fn) / n if n > 0 else 0
            p_not_actual = (tn + fp) / n if n > 0 else 0
            p_expected = (p_severe_pred * p_severe_actual) + (p_not_pred * p_not_actual)
            kappa = (p_observed - p_expected) / (1 - p_expected) if (1 - p_expected) > 0 else 0

            print(f"\n📈 CLASSIFICATION METRICS")
            print(f"   {'Accuracy:':<25} {accuracy:.1%} ({tp+tn}/{n})")
            print(f"   {'Sensitivity (Recall):':<25} {sensitivity:.1%} ({tp}/{tp+fn}) — detects SEVERE correctly")
            print(f"   {'Specificity:':<25} {specificity:.1%} ({tn}/{tn+fp}) — detects NOT_SEVERE correctly")
            print(f"   {'Precision (PPV):':<25} {precision:.1%} ({tp}/{tp+fp}) — when it says SEVERE, is it right?")
            print(f"   {'F1 Score:':<25} {f1:.3f}")
            print(f"   Cohen's Kappa:          {kappa:.3f}")

            # Kappa interpretation
            if kappa < 0:
                kappa_interp = "Less than chance"
            elif kappa < 0.21:
                kappa_interp = "Slight agreement"
            elif kappa < 0.41:
                kappa_interp = "Fair agreement"
            elif kappa < 0.61:
                kappa_interp = "Moderate agreement"
            elif kappa < 0.81:
                kappa_interp = "Substantial agreement"
            else:
                kappa_interp = "Almost perfect agreement"
            print(f"   {'Kappa interpretation:':<25} {kappa_interp}")

            # ── 7. Patient-by-Patient Comparison ──────────────
            print(f"\n" + "=" * 60)
            print("🔍 PATIENT-BY-PATIENT COMPARISON")
            print("=" * 60)
            print(f"\n{'Patient':<12} {'Pipeline':<15} {'Ground Truth':<15} {'Match':<8} {'Dr.A':<15} {'Dr.B':<15}")
            print("-" * 80)

            for _, row in merged.iterrows():
                pipeline = row["PIPELINE_BINARY"]
                gt = row["GROUND_TRUTH"]
                match = "✅" if pipeline == gt else "❌"
                dr_a = row.get("ASSESSOR_A_BINARY", "?")
                dr_b = row.get("ASSESSOR_B_BINARY", "?")
                print(f"{row['patient_short']:<12} {pipeline:<15} {gt:<15} {match:<8} {dr_a:<15} {dr_b:<15}")

            # ── 8. Individual Assessor Accuracy ───────────────
            print(f"\n" + "=" * 60)
            print("🔍 INDIVIDUAL ASSESSOR vs GROUND TRUTH")
            print("=" * 60)

            if "ASSESSOR_A_BINARY" in merged.columns:
                merged["DR_A_UPPER"] = merged["ASSESSOR_A_BINARY"].str.strip().str.upper()
                dr_a_correct = (merged["DR_A_UPPER"] == merged["GROUND_TRUTH"]).sum()
                dr_a_acc = dr_a_correct / len(merged)
                print(f"\n   Dr. A (GPT-4o) accuracy:    {dr_a_acc:.1%} ({dr_a_correct}/{len(merged)})")

                # Dr. A Kappa
                tp_a = ((merged["DR_A_UPPER"] == "SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()
                tn_a = ((merged["DR_A_UPPER"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
                fp_a = ((merged["DR_A_UPPER"] == "SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
                fn_a = ((merged["DR_A_UPPER"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()
                p_obs_a = (tp_a + tn_a) / n
                p_exp_a = ((tp_a+fp_a)/n * (tp_a+fn_a)/n) + ((tn_a+fn_a)/n * (tn_a+fp_a)/n)
                kappa_a = (p_obs_a - p_exp_a) / (1 - p_exp_a) if (1 - p_exp_a) > 0 else 0
                print(f"   Dr. A Cohen's Kappa:        {kappa_a:.3f}")

            if "ASSESSOR_B_BINARY" in merged.columns:
                merged["DR_B_UPPER"] = merged["ASSESSOR_B_BINARY"].str.strip().str.upper()
                dr_b_correct = (merged["DR_B_UPPER"] == merged["GROUND_TRUTH"]).sum()
                dr_b_acc = dr_b_correct / len(merged)
                print(f"\n   Dr. B (DeepSeek) accuracy:  {dr_b_acc:.1%} ({dr_b_correct}/{len(merged)})")

                tp_b = ((merged["DR_B_UPPER"] == "SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()
                tn_b = ((merged["DR_B_UPPER"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
                fp_b = ((merged["DR_B_UPPER"] == "SEVERE") & (merged["GROUND_TRUTH"] == "NOT_SEVERE")).sum()
                fn_b = ((merged["DR_B_UPPER"] == "NOT_SEVERE") & (merged["GROUND_TRUTH"] == "SEVERE")).sum()
                p_obs_b = (tp_b + tn_b) / n
                p_exp_b = ((tp_b+fp_b)/n * (tp_b+fn_b)/n) + ((tn_b+fn_b)/n * (tn_b+fp_b)/n)
                kappa_b = (p_obs_b - p_exp_b) / (1 - p_exp_b) if (1 - p_exp_b) > 0 else 0
                print(f"   Dr. B Cohen's Kappa:        {kappa_b:.3f}")

            # ── 9. Inter-Assessor Agreement ───────────────────
            if "DR_A_UPPER" in merged.columns and "DR_B_UPPER" in merged.columns:
                print(f"\n" + "=" * 60)
                print("🤝 INTER-ASSESSOR AGREEMENT (Dr. A vs Dr. B)")
                print("=" * 60)

                agree = (merged["DR_A_UPPER"] == merged["DR_B_UPPER"]).sum()
                agree_pct = agree / len(merged)
                print(f"   Agreement rate: {agree_pct:.1%} ({agree}/{len(merged)})")

                # Inter-assessor Kappa
                tp_ab = ((merged["DR_A_UPPER"] == "SEVERE") & (merged["DR_B_UPPER"] == "SEVERE")).sum()
                tn_ab = ((merged["DR_A_UPPER"] == "NOT_SEVERE") & (merged["DR_B_UPPER"] == "NOT_SEVERE")).sum()
                fp_ab = ((merged["DR_A_UPPER"] == "SEVERE") & (merged["DR_B_UPPER"] == "NOT_SEVERE")).sum()
                fn_ab = ((merged["DR_A_UPPER"] == "NOT_SEVERE") & (merged["DR_B_UPPER"] == "SEVERE")).sum()
                p_obs_ab = (tp_ab + tn_ab) / n
                p_exp_ab = ((tp_ab+fp_ab)/n * (tp_ab+fn_ab)/n) + ((tn_ab+fn_ab)/n * (tn_ab+fp_ab)/n)
                kappa_ab = (p_obs_ab - p_exp_ab) / (1 - p_exp_ab) if (1 - p_exp_ab) > 0 else 0
                print(f"   Inter-assessor Kappa: {kappa_ab:.3f}")

                # Where they disagree
                disagree = merged[merged["DR_A_UPPER"] != merged["DR_B_UPPER"]]
                if len(disagree) > 0:
                    print(f"\n   Disagreements ({len(disagree)}):")
                    for _, row in disagree.iterrows():
                        gt = row["GROUND_TRUTH"]
                        final = row["PIPELINE_BINARY"]
                        print(f"     {row['patient_short']}: Dr.A={row['DR_A_UPPER']}, Dr.B={row['DR_B_UPPER']} → Final={final} (GT={gt})")

            # ── 10. Error Analysis ────────────────────────────
            misclassified = merged[merged["PIPELINE_BINARY"] != merged["GROUND_TRUTH"]]
            if len(misclassified) > 0:
                print(f"\n" + "=" * 60)
                print(f"❌ MISCLASSIFIED PATIENTS ({len(misclassified)})")
                print("=" * 60)
                for _, row in misclassified.iterrows():
                    print(f"\n   Patient: {row['patient_short']}")
                    print(f"   Pipeline said: {row['PIPELINE_BINARY']}")
                    print(f"   Ground truth:  {row['GROUND_TRUTH']}")
                    print(f"   Dr. A said:    {row.get('ASSESSOR_A_BINARY', '?')}")
                    print(f"   Dr. B said:    {row.get('ASSESSOR_B_BINARY', '?')}")
                    print(f"   Agreement:     {row.get('AGREEMENT', '?')}")
                    print(f"   Confidence:    {row.get('CONFIDENCE', '?')}")
            else:
                print(f"\n🎉 No misclassifications! (but n={n} is small — wait for full 100)")

            # ── 11. Save evaluation ───────────────────────────
            eval_dir = f"{OUTPUT_DIR}/evaluation"
            os.makedirs(eval_dir, exist_ok=True)
            merged.to_csv(f"{eval_dir}/evaluation_results.csv", index=False)

            summary = {
                "n_evaluated": n,
                "accuracy": round(accuracy, 4),
                "sensitivity": round(sensitivity, 4),
                "specificity": round(specificity, 4),
                "precision": round(precision, 4),
                "f1_score": round(f1, 4),
                "cohens_kappa": round(kappa, 4),
                "kappa_interpretation": kappa_interp,
                "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn),
            }

            import json
            with open(f"{eval_dir}/evaluation_summary.json", "w") as f:
                json.dump(summary, f, indent=2)

            print(f"\n💾 Evaluation saved to {eval_dir}/")
            print(f"\n⚠️ NOTE: These are PRELIMINARY results based on {n}/100 patients.")
            print(f"   Re-run this cell after completing all 100 for definitive metrics.")